# AMPS I · Week 03 — Lab
## Mesh topology, geometry, and the DOF map

| | |
|---|---|
| **Lecture sections** | Part B (topology, geometry) · Part C (DOF map, vector/mixed, parallel) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | number and connect mesh entities · find the sign of $\det J$ · read cell-node maps for P1–P3 · count DOFs of CG, DG, vector and mixed spaces · look at owned and ghost DOFs in parallel |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx`
are available. Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Entities and adjacency  *(lecture §B1–B2)*
A 2×2 mesh: count the entities, read the adjacencies, and check the local convention "edge $i$ is opposite vertex $i$".

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
from dolfinx import mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 2, 2)
tdim = msh.topology.dim
for d in range(tdim + 1):
    msh.topology.create_entities(d)
n = [msh.topology.index_map(d).size_local for d in range(tdim + 1)]
print(f"vertices {n[0]}, edges {n[1]}, cells {n[2]}   V - E + F = {n[0] - n[1] + n[2]}")

for d1, d2 in ((2, 0), (2, 1), (1, 0)):
    msh.topology.create_connectivity(d1, d2)
c2v = msh.topology.connectivity(2, 0)
c2e = msh.topology.connectivity(2, 1)
e2v = msh.topology.connectivity(1, 0)
print("type:", type(c2v).__name__, "→ use .links(i), .array, .offsets")
for c in range(3):
    print(f"cell {c}: vertices {c2v.links(c).tolist()}  edges {c2e.links(c).tolist()}")

ok = all(set(e2v.links(c2e.links(c)[i])) == set(c2v.links(c)) - {c2v.links(c)[i]}
         for c in range(n[2]) for i in range(3))
print("local edge i is opposite local vertex i in every cell:", ok)

np.savez("/tmp/fnx/w03_mesh.npz", x=msh.geometry.x[:, :2], c2v=c2v.array.reshape(-1, 3),
         e2v=e2v.array.reshape(-1, 2))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/w03_mesh.npz")
x, c2v, e2v = d["x"], d["c2v"], d["e2v"]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for c, cell in enumerate(c2v):
    P = x[cell]
    ax.fill(P[:, 0], P[:, 1], fill=False, ec="0.4")
    ax.text(*P.mean(0), f"c{c}", color="C2", ha="center", va="center", weight="bold")
for e, (a, b) in enumerate(e2v):
    ax.text(*(0.5 * (x[a] + x[b])), f"e{e}", color="C1", ha="center", va="center", fontsize=9,
            bbox=dict(fc="w", ec="C1", boxstyle="round,pad=0.15"))
for v, p in enumerate(x):
    ax.text(*p, f"v{v}", color="w", ha="center", va="center", fontsize=9,
            bbox=dict(fc="C0", ec="C0", boxstyle="circle,pad=0.2"))
ax.set_aspect("equal"); ax.axis("off"); plt.show()

> **✏️ Your turn (5 min).** On a 3×3 mesh, classify the edges with the upward adjacency $\mathrm{Adj}_{1,2}$:
> an edge with one cell is on the boundary. Compare the boundary count with `mesh.exterior_facet_indices`.

In [ ]:
%%fenicsx -np 1
# TODO: 3×3 mesh; create_connectivity(1, 2); count edges with 1 and 2 cells; compare with exterior_facet_indices

## 2 · Geometry and the sign of $\det J$  *(lecture §B3)*
`geometry.x` holds the node coordinates; `geometry.dofmaps[0][c]` lists the nodes of cell $c$ (a NumPy array).

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
from dolfinx import mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 4, 4)
x = msh.geometry.x[:, :2]
P = x[msh.geometry.dofmaps[0]]                      # (n_cells, 3, 2): vertex coordinates per cell
a, b = P[:, 1] - P[:, 0], P[:, 2] - P[:, 0]
detJ = a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
print(f"cells {len(detJ)}: det J > 0 in {np.sum(detJ > 0)}, det J < 0 in {np.sum(detJ < 0)}")
print(f"sum det J / 2 = {detJ.sum() / 2:.3f}    sum |det J| / 2 = {np.abs(detJ).sum() / 2:.3f}")

> **✏️ Your turn (3 min).** Repeat with `diagonal=mesh.DiagonalType.crossed`. How many cells have $\det J<0$?
> What would an assembly routine that uses $\det J$ instead of $|\det J|$ compute for $\int_\Omega 1\,dx$?

In [ ]:
%%fenicsx -np 1
# TODO: crossed diagonal; count det J < 0; total signed and unsigned area

## 3 · The DOF map  *(lecture §C1–C3)*

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import basix, basix.ufl
from dolfinx import fem, mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 2, 2)
for k in (1, 2, 3):
    V = fem.functionspace(msh, ("Lagrange", k))
    dm = V.dofmap
    c0, c1 = dm.cell_dofs(0), dm.cell_dofs(1)
    per_entity = [len(dm.dof_layout.entity_dofs(d, 0)) for d in range(3)]
    print(f"P{k}: {dm.index_map.size_global} DOFs; per vertex/edge/cell {per_entity}")
    print(f"     cell 0 {c0.tolist()}\n     cell 1 {c1.tolist()}\n     shared {np.intersect1d(c0, c1).tolist()}")

V2 = fem.functionspace(msh, ("Lagrange", 2))
np.savez("/tmp/fnx/w03_p2dofs.npz", xy=V2.tabulate_dof_coordinates()[:, :2],
         cells=msh.geometry.x[:, :2][msh.geometry.dofmaps[0]])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/w03_p2dofs.npz")
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for P in d["cells"]:
    ax.fill(P[:, 0], P[:, 1], fill=False, ec="0.5")
for i, p in enumerate(d["xy"]):
    ax.text(*p, str(i), ha="center", va="center", fontsize=9, bbox=dict(boxstyle="circle,pad=0.25", fc="w", ec="C1"))
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("P2 global DOF numbers"); plt.show()

Look at cell 1 for P3: its first edge DOFs are listed as `12, 11`, not `11, 12`. DOLFINx has reversed them so that both
cells sharing that edge put DOF 11 at the same point (lecture §C2).

> **✏️ Your turn (7 min).** Check the dimension formulas on the 4×4 mesh for $k=1,\dots,4$:
> CG $\dim V_h = n_V+(k-1)n_E+\tfrac{(k-1)(k-2)}{2}n_T$ and DG $\dim V_h=n_T\,\tfrac{(k+1)(k+2)}{2}$.
> Take $n_V,n_E,n_T$ from the topology, not from a formula in $N$.

In [ ]:
%%fenicsx -np 1
# TODO: n_V, n_E, n_T from the topology of a 4×4 mesh; compare both formulas with size_global for k = 1..4

## 4 · Vector and mixed spaces  *(lecture §C4)*

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import basix.ufl
from dolfinx import fem, mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 4, 4)
Vv = fem.functionspace(msh, ("Lagrange", 2, (2,)))
im, bs = Vv.dofmap.index_map, Vv.dofmap.index_map_bs
print(f"vector P2: size_global {im.size_global}, block size {bs} → {im.size_global * bs} scalar DOFs")
print("cell_dofs(0) are block numbers:", Vv.dofmap.cell_dofs(0).tolist())

P2v = basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))
P1 = basix.ufl.element("Lagrange", "triangle", 1)
W = fem.functionspace(msh, basix.ufl.mixed_element([P2v, P1]))
W0, maps0 = W.sub(0).collapse()
W1, maps1 = W.sub(1).collapse()
print(f"Taylor–Hood: W has {W.dofmap.index_map.size_global} DOFs (bs {W.dofmap.index_map_bs}), "
      f"{len(W.dofmap.cell_dofs(0))} per cell")
print(f"  velocity {W0.dofmap.index_map.size_global * W0.dofmap.index_map_bs}, pressure {W1.dofmap.index_map.size_global}")
print("  collapse() returns", type(maps0).__name__, "of", len(maps0), "array; pressure DOFs inside W:", maps1[0][:6].tolist(), "...")

> **✏️ Your turn (5 min).** *Before running anything*, compute by hand the number of DOFs of the Taylor–Hood space
> (P2 velocity in 2D + P1 pressure) on an 8×8 mesh. Then check it with DOLFINx.

In [ ]:
%%fenicsx -np 1
# TODO: your hand count in a comment, then the DOLFINx check on an 8×8 mesh

## 5 · Owned and ghost DOFs  *(lecture §C5)*
The same space on 2 ranks. Each rank prints its own numbers.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from dolfinx import fem, mesh

comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 4, 4)
for fam in ("Lagrange", "DG"):
    im = fem.functionspace(msh, (fam, 1)).dofmap.index_map
    total = comm.allreduce(im.size_local, op=MPI.SUM)
    print(f"rank {comm.rank} {fam:8s} P1: owned {im.size_local:3d}, ghosts {im.num_ghosts:3d}, "
          f"size_global {im.size_global}, sum of owned over ranks {total}", flush=True)

> **✏️ Your turn (3 min).** A DG DOF belongs to one cell only. Why does the DG space still have ghost DOFs?

> *your answer*

## 6 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11 code for a %%fenicsx -np 2 cell. Create a 6×6 unit square mesh. On every rank, compute the
area of each owned cell and the number of owned boundary facets. Print, on rank 0 only, the total area, the total
number of cells, and the total number of boundary facets. Then save the cell vertex coordinates of rank 0 to
/tmp/fnx/m.npz and plot them in a separate ordinary cell (NumPy + matplotlib only).
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer, not fixes

- [ ] The output has no warnings (read stderr too)
- [ ] The total area is exactly 1
- [ ] The cell count is $2N^2=72$ and the boundary-facet count is $4N=24$
- [ ] Each number is printed once, not once per rank
- [ ] Changing the magic to `-np 1` gives the same numbers